In [ ]:
# The GFG articles correctly describe buffer memory as storing the entire conversation 
# and window memory as retaining only recent turns.

# But modern LangChain doesn't require separate legacy classes for this.

# The modern question is:

# How much of the conversation should I keep in the agent's state/context?

### Full Buffer Concept

In [ ]:
# Conceptually:

# Message 1
# Message 2
# Message 3
# Message 4
# Message 5
# Message 6
# Message 7
# ...

# Everything is retained.




# Problem:

# history ↑
# tokens ↑
# cost ↑
# latency ↑




# Eventually:

# context window exceeded



# Therefore:

# FULL HISTORY
#      ↓
# may become too large
#      ↓
# TRIM / DELETE / SUMMARIZE



# The current LangChain docs explicitly recommend trimming, deleting, or 
# summarizing messages when conversations become too long.

### Window Memory Concept

In [ ]:
# Suppose we want only the last 4 messages.

# Old:

# 1
# 2
# 3
# 4
# 5
# 6
# 7
# 8

# Keep:

# 5
# 6
# 7
# 8

# This is what the old ConversationBufferWindowMemory(k=...) article was trying to teach.

# Modern implementation can manipulate agent state instead of using that deprecated-style memory class.

### Message Trimming

In [ ]:
# Current LangChain provides message-management patterns for this.

# For example, a middleware can trim old messages before the model call.

In [1]:
from typing import Any

from langchain.agents import create_agent, AgentState
from langchain.agents.middleware import before_model
from langchain.messages import RemoveMessage
from langgraph.graph.message import REMOVE_ALL_MESSAGES
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.runtime import Runtime


@before_model
def trim_messages(
    state: AgentState,
    runtime: Runtime
) -> dict[str, Any] | None:

    messages = state["messages"]

    if len(messages) <= 5:
        return None

    recent_messages = messages[-4:]

    return {
        "messages": [
            RemoveMessage(id=REMOVE_ALL_MESSAGES),
            *recent_messages
        ]
    }

In [ ]:
agent = create_agent(
    model="google_genai:gemini-3.5-flash-lite",
    tools=[],
    middleware=[trim_messages],
    checkpointer=InMemorySaver()
)

In [ ]:
# The important idea is:

# Agent receives huge history
#           ↓
# middleware
#           ↓
# remove old messages
#           ↓
# Gemini receives manageable context

# The current LangChain documentation shows this middleware-based approach for message trimming.

### Token-Based Trimming

In [4]:
# Counting messages isn't always enough.





# One message could be:

# "Hi"



# while another could contain:

# 20,000 tokens

# Therefore production systems often consider tokens, not simply number of messages.



# Conceptually:

# Keep history
# until


# token_count ≈ allowed_limit




# LangChain currently provides message trimming utilities for this purpose.




# You should remember:

# Message count ≠ token count.

### Delete Messages

In [ ]:
# Modern LangGraph state supports:

from langchain.messages import RemoveMessage

# For example:

def delete_old_messages(state):
    messages = state["messages"]

    if len(messages) > 6:
        return {
            "messages": [
                RemoveMessage(id=m.id)
                for m in messages[:-6]
            ]
        }

    return None

# The current docs specifically show RemoveMessage 
# for deleting individual messages or clearing the entire message history.